# Bonus — Extraccion con IA sobre un POT real (Rionegro)

Este notebook usa un **fragmento real** del Plan de Ordenamiento Territorial de Rionegro
(Decreto 124 de 2018, paginas 305-315 — Condiciones de Localizacion de Usos Industriales
en Suelo Rural), para demostrar el flujo de IA sobre documentos que propusimos en el
caso de cierre de Grupo Argos.

**Nota importante:** el documento original tiene 515 paginas. Para esta demo usamos
solo el fragmento relevante (extraido con `pypdf`) para que la ejecucion sea agil.
En el caso real, este mismo flujo se repetiria por cada capitulo relevante de cada
municipio scrapeado.

In [0]:
%pip install pypdf
dbutils.library.restartPython()

dbutils.library.restartPython()

In [0]:
CATALOG = "laboratory_sap_dev"
SCHEMA  = "sap_course"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/curso_databricks/pot_demo"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Leyendo desde: {VOLUME_PATH}")

## Paso 1 — Bronze: el PDF como binaryFile

Igual que hicimos con Auto Loader para los CSV de SAP, un PDF entra al Lakehouse
como un archivo binario crudo.

In [0]:
df_bronze = (spark.read.format("binaryFile")
    .load(f"{VOLUME_PATH}/Rionegro_POT_Uso_Industrial_extracto.pdf"))

df_bronze.select("path", "length", "modificationTime").show(truncate=False)

(df_bronze.write.mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.pot_documentos_bronze"))

## Paso 2 — El problema: el PDF YA tiene una capa de texto, y es basura

Este documento fue escaneado (Creator: Canon) y trae un OCR de mala calidad ya
incrustado. Si extraes el texto de forma ingenua (ej. `pypdf`, `pdftotext`), esto
es lo que obtienes -- literalmente ilegible para un `ai_extract` confiable:

```
¡F¡i
D:---,t
l(onegro
Todos¿.É

Toreo de
```

(Ese fragmento debería decir "Rionegro" y "Tarea de Todos", el lema del municipio.)

**Por eso NO usamos el texto embebido.** Vamos directo a `ai_parse_document`, que
usa un modelo de vision sobre la imagen renderizada de cada pagina, no la capa de
texto rota.

In [0]:
# Comprobacion rapida del problema (ejecutar solo si quieres ver la evidencia en vivo)
import io
from pypdf import PdfReader

pdf_bytes = df_bronze.select("content").first()["content"]
reader = PdfReader(io.BytesIO(pdf_bytes))
texto_crudo = reader.pages[0].extract_text()

print("TEXTO EMBEBIDO (OCR original, de mala calidad):")
print(texto_crudo[:400])

## Paso 3 — `ai_parse_document`: extraccion real via vision

Esta funcion nativa de Databricks SQL procesa el documento pagina por pagina usando
un modelo con capacidad de vision, ignorando la capa de texto rota.

In [0]:
parsed_df = spark.sql(f'''
    SELECT
        path,
        ai_parse_document(content) AS documento_parseado
    FROM {CATALOG}.{SCHEMA}.pot_documentos_bronze
''')

parsed_df.createOrReplaceTempView("pot_parseado_tmp")

# El resultado es un VARIANT estructurado: paginas, elementos, texto por bloque
spark.sql('''
    SELECT
        path,
        documento_parseado:document:pages AS paginas
    FROM pot_parseado_tmp
''').display()

## Paso 4 — `ai_extract`: estructurar los campos que necesita Grupo Argos

Del texto ya limpio, extraemos exactamente los campos que definimos en el caso de
cierre: clasificacion de impacto ambiental/urbanistico y condiciones de localizacion
para uso industrial -- el insumo que despues se cruza con DANE, IGAC y Camacol.

In [0]:
extraido_df = spark.sql(f'''
    WITH texto AS (
        SELECT
            path,
            array_join(
                transform(
                    CAST(documento_parseado:document:elements AS ARRAY<VARIANT>),
                    e -> CAST(e:content AS STRING)
                ),
                '\n'
            ) AS texto_completo
        FROM pot_parseado_tmp
    ),
    extraido AS (
        SELECT
            path,
            ai_extract(
                texto_completo,
                array(
                    'municipio',
                    'categoria_uso_suelo',
                    'clasificacion_impacto_ambiental',
                    'condiciones_localizacion',
                    'requiere_licencia_urbanistica'
                )
            ) AS campos_extraidos
        FROM texto
    )
    SELECT
        path,
        campos_extraidos.municipio,
        campos_extraidos.categoria_uso_suelo,
        campos_extraidos.clasificacion_impacto_ambiental,
        campos_extraidos.condiciones_localizacion,
        campos_extraidos.requiere_licencia_urbanistica
    FROM extraido
''')

extraido_df.display()

## Paso 5 — Silver: tabla lista para el modelo de scoring

Una fila estructurada por municipio, lista para el JOIN con las fuentes externas
(DANE, IGAC, Camacol) que propusimos en la arquitectura del caso Grupo Argos.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.pot_uso_industrial_silver AS
    WITH texto AS (
        SELECT
            path,
            array_join(
                transform(
                    CAST(documento_parseado:document:elements AS ARRAY<VARIANT>),
                    e -> CAST(e:content AS STRING)
                ),
                '\n'
            ) AS texto_completo
        FROM pot_parseado_tmp
    ),
    extraido AS (
        SELECT
            path,
            ai_extract(
                texto_completo,
                array(
                    'municipio',
                    'clasificacion_impacto_ambiental',
                    'condiciones_localizacion',
                    'requiere_licencia_urbanistica'
                )
            ) AS campos_extraidos
        FROM texto
    )
    SELECT
        path,
        campos_extraidos.municipio                       AS municipio,
        'Antioquia'                                        AS departamento,
        'Industrial'                                       AS categoria_uso_suelo,
        campos_extraidos.clasificacion_impacto_ambiental  AS clasificacion_impacto,
        campos_extraidos.condiciones_localizacion         AS condiciones_localizacion,
        campos_extraidos.requiere_licencia_urbanistica    AS requiere_licencia,
        current_timestamp()                                AS _procesado_at
    FROM extraido
""")

display(spark.table(f"{CATALOG}.{SCHEMA}.pot_uso_industrial_silver"))

## Cierre — como escala esto al caso completo

Para el caso real de Grupo Argos, este mismo flujo (Bronze binaryFile -> `ai_parse_document`
-> `ai_extract` -> Silver estructurado) se repite por cada ciudad intermedia scrapeada
de Asointermedias, agregando una columna `municipio` distinta por documento. Con esa
tabla Silver ya estructurada, el siguiente paso es el `JOIN` en Gold contra:

- **DANE** — proyecciones de poblacion
- **IGAC** — avaluos catastrales
- **Camacol** — licencias de construccion vigentes

...tal como quedo planteado en la diapositiva de arquitectura del caso de cierre.